# Slice R&D Benchmark (PFF→in-memory L1/L2)

Measures PFF-slice→L1→features→inference vs full materialization, across slice sizes and decimation factors.

In [ ]:
from pathlib import Path

from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.paths import REPO_ROOT

# ── Configure ───────────────────────────────────────────────────────────────────────────────
OBS_DIR = Path("/path/to/obs.pffd")  # replace
DP = "dp_img16.bpp_2.module_1"  # replace with actual product name

results: list[BenchResult] = []

## §1 Full materialization baseline (convert + calibrate)

In [ ]:
from panoseti_analysis.adapters.calibrate import run_calibrate
from panoseti_analysis.adapters.convert import run_convert

OUT_BASE = Path("/tmp/slice_rnd_bench")
CONVERT_OUT = OUT_BASE / "convert"
CALIBRATE_OUT = OUT_BASE / "calibrate"

with stage_timer("full_pipeline_baseline", bytes_in=0) as r:
    records_convert = run_convert(OBS_DIR, CONVERT_OUT, checksum=False)

    # Find the L0 store for this product and module
    l0_stores = list(CONVERT_OUT.glob("**/*.L0.zarr"))
    if l0_stores:
        l0_store = l0_stores[0]
        records_calibrate = run_calibrate(l0_store, CALIBRATE_OUT)
        out_bytes = sum(f.stat().st_size for f in CALIBRATE_OUT.rglob("*") if f.is_file())
        r.bytes_out = out_bytes

results.append(r)
print(summarize(results))

## §2 In-memory slice (slice_to_l1)

In [ ]:
from panoseti_analysis.adapters.slice_driver import slice_to_l1
from panoseti_analysis.io.pff import open_pff_product

seq = open_pff_product(OBS_DIR, DP, module=1)

# Benchmark slice_to_l1 at different decimation factors
for decimate in [1, 5, 10, 50]:
    with stage_timer(f"slice_to_l1_decimate_{decimate}", bytes_in=0) as r:
        ds_l1 = slice_to_l1(seq, frame_range=(0, 5000), decimate=decimate)
    r.bytes_out = ds_l1["median_subtracted"].nbytes
    results.append(r)

print(summarize(results))

## §3 In-memory L2 (slice_to_l2_cloud)

In [ ]:
from panoseti_analysis.adapters.slice_driver import slice_to_l2_cloud
from panoseti_analysis.algorithms.cloud_detector import CloudDetectionV2
from panoseti_analysis.config.models import CloudInferParams
from panoseti_analysis.io.models import load_classifier

MODEL_PT = REPO_ROOT / "ml/cloud-detection/models/cloud_detector_v2_legacy.pt"
INFER_PARAMS = CloudInferParams(cadence_s=10.0, window_s=60.0, n_stack=10)

if not MODEL_PT.exists():
    print(f"Model not found: {MODEL_PT} — skipping slice_to_l2_cloud benchmark.")
else:
    model, bundle = load_classifier(MODEL_PT, model=CloudDetectionV2())
    model.eval()

    for decimate in [1, 10, 50]:
        with stage_timer(f"slice_to_l2_cloud_decimate_{decimate}", bytes_in=0) as r:
            ds_l2 = slice_to_l2_cloud(
                seq,
                model,
                frame_range=(0, 5000),
                decimate=decimate,
                infer_params=INFER_PARAMS,
            )
        r.bytes_out = ds_l2["cloud_score"].nbytes
        results.append(r)

    print(summarize(results))

## §4 Summary

In [ ]:
print(summarize(results))